# AnimalPoseTracker Demo

Run a 10-epoch AnimalRTPose-N workflow on Colab: training, validation, test prediction from `.pt`, ONNX export, and test prediction from ONNX. The dataset stays in `MyDrive/datasets/trimouse`; the repository and project stay under `MyDrive/wux024`. Enable a GPU runtime before running.

Set `RUN_MODE` to `new` or `load`. In load mode, set `PROJECT_PATH` to the existing project folder in Drive; its saved configs and checkpoint are reused.

The Colab runtime is a cloud Linux machine. Repository update, installation, training, validation, prediction, and export run in Bash command cells. Project creation and configuration use `animalpose-cli create`. Run cells from the top; the repository cell updates `preview` and prints the commit used.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# @title AnimalPoseTracker run options {display-mode: "form", run: "auto", single-column: true}
# @markdown Choose a mode, then run the notebook cells. `new` creates and trains; `load` reuses a Drive project.
RUN_MODE = "new" # @param ["new", "load"]
# @markdown Required for `load`: the existing project folder in Google Drive.
PROJECT_PATH = "" # @param {type:"string"}
# @markdown Optional override; blank uses `PROJECT_PATH/runs/train/weights/best.pt`.
CHECKPOINT_PATH = "" # @param {type:"string"}


In [ ]:
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/wux024/AnimalPoseTracker.git"
BRANCH = "preview"
REPO_DIR = Path("/content/drive/MyDrive/wux024/AnimalPoseTracker")
PROJECTS_ROOT = REPO_DIR.parent
DATA_SOURCE_ROOT = Path("/content/drive/MyDrive/datasets/trimouse")
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
PREDICT_SPLIT = "test"
EXPORT_DIR = f"runs/export-{RUN_ID}"

if RUN_MODE == "load":
    PROJECT_DIR = Path(PROJECT_PATH).expanduser().resolve()
    BEST_CHECKPOINT = (
        Path(CHECKPOINT_PATH).expanduser().resolve()
        if CHECKPOINT_PATH.strip()
        else PROJECT_DIR / "runs/train/weights/best.pt"
    )
else:
    PROJECT_DIR = PROJECTS_ROOT / f"trimouse-colab-AnimalRTPose-N-{RUN_ID}"
    BEST_CHECKPOINT = PROJECT_DIR / "runs/train/weights/best.pt"

REPO_DIR_STR = str(REPO_DIR)
PROJECT_DIR_STR = str(PROJECT_DIR)
%env REPO_URL={REPO_URL}
%env BRANCH={BRANCH}
%env REPO_DIR={REPO_DIR_STR}
%env PROJECTS_ROOT={PROJECTS_ROOT}
%env DATA_SOURCE_ROOT={DATA_SOURCE_ROOT}
%env RUN_ID={RUN_ID}
%env PROJECT_DIR={PROJECT_DIR_STR}
%env BEST_CHECKPOINT={BEST_CHECKPOINT}
%env RUN_MODE={RUN_MODE}
%env PREDICT_SPLIT={PREDICT_SPLIT}
%env EXPORT_DIR={EXPORT_DIR}


In [ ]:
%%bash
set -e
mkdir -p "$(dirname "$REPO_DIR")"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --depth 1 --single-branch --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
else
  git -C "$REPO_DIR" pull --ff-only origin "$BRANCH"
fi
git -C "$REPO_DIR" log -1 --oneline


In [ ]:
%%bash
set -e
python -m pip install -e "$REPO_DIR"
python -m pip install pycocotools onnx onnxsim onnxruntime


In [ ]:
%%bash
set -e
cd "$REPO_DIR"
if [ "$RUN_MODE" = "new" ]; then
  animalpose-cli create \
    --dataset animalposetracker/cfg/datasets/trimouse.yaml \
    --dataset-root "$DATA_SOURCE_ROOT" \
    --workspace "$PROJECTS_ROOT" \
    --name trimouse --worker colab --model AnimalRTPose --scale N \
    --date "$RUN_ID" --epochs 10 --batch 4 --imgsz 640 --workers 2 --device 0
fi


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
if [ "$RUN_MODE" = "new" ]; then
  animalpose-cli train --config configs/other.yaml
fi


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
animalpose-cli val --config configs/other.yaml --weights "$BEST_CHECKPOINT"


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
animalpose-cli predict --config configs/other.yaml --weights "$BEST_CHECKPOINT" --split "$PREDICT_SPLIT"


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
animalpose-cli export --config configs/other.yaml --weights "$BEST_CHECKPOINT" --format onnx --output-dir "$EXPORT_DIR"


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
animalpose-cli predict --config configs/other.yaml --weights "$PROJECT_DIR/$EXPORT_DIR/best.onnx" --split "$PREDICT_SPLIT"
